In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from pydantic import BaseModel
from typing import List
from dotenv import load_dotenv
from langchain_chroma import Chroma
import os
from collections import defaultdict

In [2]:
load_dotenv()

True

In [3]:
class state(BaseModel):
    queries:List[str]

In [4]:
llm=ChatGoogleGenerativeAI(model="gemini-2.5-pro")
structured_llm=llm.with_structured_output(schema=state)

In [5]:
query="How does Tesla make money"

In [6]:
prompt=f"""Generate three variation of the user query, for multi query retrival in RAG.\n
USER QUERY:\n
{query}"""

In [7]:
response=structured_llm.invoke(prompt)
quary_variations=response.queries

In [8]:
response.queries

["What are Tesla's primary revenue streams?",
 "How does Tesla's business model generate profit?",
 "Breakdown of Tesla's income sources, including automotive, energy, and services."]

In [9]:
current_dir=os.getcwd()
persistant_dir=os.path.join(current_dir,"database","Chroma_db")

In [10]:
embedding=GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

db=Chroma(
    persist_directory=persistant_dir,
    embedding_function=embedding,
    collection_metadata={"hnsw:space":"cosine"}
)

retriver=db.as_retriever(
    search_type="mmr",
    search_kwargs={"k":3,"fetch_k":10, "lambda_mult":0.5}
)

In [11]:
all_retrivals=[]
for quary in quary_variations:
    retrival=retriver.invoke(quary)
    all_retrivals.append(retrival)

In [12]:
all_retrivals

[[Document(id='f81ec15d-d5ca-4068-9440-3b4ff62de8fb', metadata={'source': 'd:\\programming\\RAG implimentation\\RAG_Practice\\2.RAG_with_chat_history\\documents\\Tesla.txt', 'filename': 'Tesla.txt'}, page_content='government pollution standards. That number has been a\nsmaller percentage of revenue for multiple quarters.[562] Automotive 87.6 89.7%\n\nEnergy Generation and Storage 10.1 10.3%\nTesla ended 2021 with $17.6 billion of cash on hand, down\n$1.8 billion from the end of 2020.[158]:\u200a31\u200a Sales by region (2024)[563]\n\nRegion Sales in billion $ Share\nIn February 2021, a 10-K filing revealed that Tesla had invested some'),
  Document(id='a91dc0e3-86ce-47a7-b658-2e7bcfbb86ec', metadata={'source': 'd:\\programming\\RAG implimentation\\RAG_Practice\\2.RAG_with_chat_history\\documents\\Tesla.txt', 'filename': 'Tesla.txt'}, page_content="Tesla was incorporated in July 2003 by Martin Eberhard and Marc\nTarpenning as Tesla Motors. Its name is a tribute to inventor and\nelectric

In [13]:
def reciprocal_rank_fusion(all_retrivals,k=60):
    rrf_socre=defaultdict(float)
    unique_chunks={}

    for query_idx, chunks in enumerate(all_retrivals,1):
        for position, chunk in enumerate(chunks,1):
            chunk_content=chunk.page_content
            position_score=1/(k + position)
            rrf_socre[chunk_content]+=position_score

            unique_chunks[chunk_content]=chunk
    

    sorted_chunks=sorted(
        [(unique_chunks[chunk_content],score) for chunk_content,score in rrf_socre.items()],
        key=lambda x:x[1], reverse=True
    )
    

    return sorted_chunks

In [14]:
sorted_chunks=reciprocal_rank_fusion(all_retrivals=all_retrivals,k=60)

In [15]:
for chunk in sorted_chunks:
    print(f"Chunk  : {chunk[0].page_content}")
    print(f"Source : {chunk[0].metadata['source']}")
    print(f"Score  : {chunk[1]}")

Chunk  : government pollution standards. That number has been a
smaller percentage of revenue for multiple quarters.[562] Automotive 87.6 89.7%

Energy Generation and Storage 10.1 10.3%
Tesla ended 2021 with $17.6 billion of cash on hand, down
$1.8 billion from the end of 2020.[158]: 31  Sales by region (2024)[563]

Region Sales in billion $ Share
In February 2021, a 10-K filing revealed that Tesla had invested some
Source : d:\programming\RAG implimentation\RAG_Practice\2.RAG_with_chat_history\documents\Tesla.txt
Score  : 0.03278688524590164
Chunk  : Tesla was incorporated in July 2003 by Martin Eberhard and Marc
Tarpenning as Tesla Motors. Its name is a tribute to inventor and
electrical engineer Nikola Tesla. In February 2004, Elon Musk led
Tesla's first funding round and became the company's chairman; in
2008, he was named chief executive officer. In 2008, the company
began production of its first car model, the Roadster sports car, followed
by the Model S sedan in 2012, the Model 